# GOMb0.01 layout A: local smoke test

Step 2 of [ocean-icechunks/hycom#17](https://github.com/ocean-icechunks/hycom/issues/17). It builds a
tiny virtual Icechunk store from six hours of 2022 in a **temporary local repository** and checks
it against the source files. The plan, the decisions and the metadata repairs R1–R20 are in
[`claude/notes/plan-issue-17.md`](../claude/notes/plan-issue-17.md). The code is in
[`gom_virtual.py`](gom_virtual.py).

What it checks:
1. Discovery and file selection over the whole bucket, with every skipped key given a reason.
2. Byte-range access to the source.
3. A **transition sample**: files from both sides of every experiment boundary and every header-size
   variant, scanned with both parsers, (a) VirtualiZarr's `HDFParser` and (b) h5py.
4. A store at two 3z chunkings (10 and 1 depths per chunk), written the way production will write
   it: a skeleton commit, then region writes per batch.
5. A **fresh read-only session**: structure, attributes, and values compared with h5py reads of
   the same files, including two kinds of gap hour.
6. The CF check (IOOS compliance checker, `cf:1.11`) on an export from the store.

Nothing is written outside a temporary directory, which the last cell removes.

In [1]:
import sys, time, tempfile, shutil, hashlib
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, zarr, icechunk, virtualizarr, h5py, obstore
import gom_virtual as gv

zarr.config.set({"async.concurrency": 128})   # the default of 10 measures the wrong thing
print("python", sys.version.split()[0])
for m in (icechunk, virtualizarr, zarr, xr, np, pd, h5py, obstore):
    print(f"{m.__name__:12s} {m.__version__}")
print("zarr async.concurrency", zarr.config.get("async.concurrency"))

python 3.13.18
icechunk     2.3.1
virtualizarr 2.8.0
zarr         3.4.2
xarray       2026.9.0
numpy        2.5.5
pandas       3.1.1
h5py         3.16.0
obstore      0.12.1
zarr async.concurrency 128


## 1. Discovery

The real method: an anonymous listing of every year directory. `select_files` keeps layout A and says why each other hourly key was left out.

In [2]:
store = gv.source_store()
t0 = time.perf_counter()
listing = gv.list_keys(store, range(2001, 2025))
files, skipped = gv.select_files(listing)
print(f"listed {len(listing):,} objects in {time.perf_counter() - t0:.0f} s")
print(f"layout A files: {len(files):,}", files.kind.value_counts().to_dict())
print(skipped.reason.value_counts().to_string())
skipped[~skipped.reason.str.startswith("layout B")]

listed 309,759 objects in 26 s
layout A files: 296,585 {'2d': 148457, '3z': 148128}
reason
layout B experiment (027/028)                           13162
too small: empty file                                       6
held back: regenerated 2026-11-10, waiting for HYCOM        5
not an hourly file name (_CORRUPTED, _COPY, stray)          1


,key,size,reason
0,2002/010_archv.2002_046_12_3z.nc,2592206800,"held back: regenerated 2026-11-10, waiting for..."
1,2002/010_archv.2002_046_16_3z.nc,2592206800,"held back: regenerated 2026-11-10, waiting for..."
2,2002/010_archv.2002_051_06_3z.nc,2592206800,"held back: regenerated 2026-11-10, waiting for..."
3,2002/010_archv.2002_051_07_3z.nc,2592206800,"held back: regenerated 2026-11-10, waiting for..."
4,2002/010_archv.2002_051_09_3z.nc,2592206800,"held back: regenerated 2026-11-10, waiting for..."
5,2006/010_archv.2006_177_18_3z.nc,49818,too small: empty file
6,2006/010_archv.2006_177_19_3z.nc,49818,too small: empty file
7,2006/010_archv.2006_177_20_3z.nc,49818,too small: empty file
8,2006/010_archv.2006_177_21_3z.nc,49818,too small: empty file
9,2006/010_archv.2006_177_22_3z.nc,49818,too small: empty file


## 2. Byte-range access

A ranged GET over the HTTPS URL that the references will use must return 206 and the HDF5 signature.

In [3]:
import urllib.request
url = gv.URL_PREFIX + "2022/023_archv.2022_100_12_3z.nc"
req = urllib.request.Request(url, headers={"Range": "bytes=0-7"})
with urllib.request.urlopen(req) as r:
    status, magic = r.status, r.read()
print(status, magic)
assert status == 206 and magic == b"\x89HDF\r\n\x1a\n"

206 b'\x89HDF\r\n\x1a\n'


## 3. Transition sample: both parsers

Both sides of the 010→023 boundary (2017-06-01 18/19Z) and the 023→026 boundary (2024-01-01
18/19Z), the first and last layout A hours, every file size seen in layout A, and the 2005 file
HYCOM interpolated with `ncra`. Both parsers must give identical offsets. The plan keeps the faster.

In [4]:
SAMPLE = ["2001/010_archv.2001_016_00", "2001/010_archv.2001_037_00", "2002/010_archv.2002_102_05",
          "2003/010_archv.2003_274_00", "2003/010_archv.2003_302_06", "2004/010_archv.2004_153_19",
          "2005/010_archv.2005_079_04", "2017/010_archv.2017_152_18", "2017/023_archv.2017_152_19",
          "2020/023_archv.2020_020_00", "2022/023_archv.2022_100_12", "2023/023_archv.2023_365_00",
          "2024/023_archv.2024_001_18", "2024/026_archv.2024_001_19", "2024/026_archv.2024_046_17",
          "2024/026_archv.2024_119_23"]
sample = files[files.key.str[:-6].isin(SAMPLE)]
assert len(sample) == 2 * len(SAMPLE)
res = {}
for method in ("hdfparser", "h5py"):
    t0 = time.perf_counter()
    res[method] = gv.scan_files(sample, method, workers=16)
    print(f"{method:9s}: {len(sample)} files in {time.perf_counter() - t0:5.1f} s")
scans, report = res["h5py"]
other = res["hdfparser"][0]
assert scans.keys() == other.keys()
assert all(np.array_equal(scans[k]["offsets"][v], other[k]["offsets"][v]) for k in scans for v in scans[k]["offsets"])
print("offsets identical between the two parsers in all", len(scans), "files that both accepted")
report.merge(sample[["key", "size"]])[["key", "size", "fingerprint", "error"]]

hdfparser: 32 files in   7.8 s


h5py     : 32 files in   4.8 s
offsets identical between the two parsers in all 31 files that both accepted


,key,size,fingerprint,error
0,2001/010_archv.2001_016_00_2d.nc,77566331,9eacfbca62,NaN
1,2001/010_archv.2001_016_00_3z.nc,2592206800,936320f7b7,NaN
2,2001/010_archv.2001_037_00_2d.nc,77566414,9eacfbca62,NaN
3,2001/010_archv.2001_037_00_3z.nc,2592206800,936320f7b7,NaN
4,2002/010_archv.2002_102_05_2d.nc,77566331,9eacfbca62,NaN
5,2002/010_archv.2002_102_05_3z.nc,2592206837,936320f7b7,NaN
6,2003/010_archv.2003_274_00_2d.nc,77566343,9eacfbca62,NaN
7,2003/010_archv.2003_274_00_3z.nc,2592206800,936320f7b7,NaN
8,2003/010_archv.2003_302_06_2d.nc,77566426,9eacfbca62,NaN
9,2003/010_archv.2003_302_06_3z.nc,2592206800,936320f7b7,NaN


What the sample shows (2026-11-19):

- **`2004/010_archv.2004_153_19_2d.nc` is chunked like layout B** (769 × 1051), although it is
  experiment 010. Both parsers reject it, so that hour is a 2d gap. It is listed for HYCOM in the plan.
- **One attribute fingerprint differs**, the 2005 file HYCOM rebuilt with `ncra` from its neighbours.
  It has `cell_methods = "MT: mean"` and an NCO `history`. Its data are fine to reference. The README lists such hours.
- **Coordinates:** the 2d `Latitude`/`Longitude` are identical everywhere. The 3z `Longitude` of 023/026 files
  differs from it by up to 3e-5°, so the store takes them from a 2d file (`gv.read_grid`). Some 3z files
  also put a `valid_range` on the coordinates, which is dropped as R7 drops it on the data.
- **The two parsers agree byte for byte.** h5py is about 2–3× faster here (both run in processes, see
  `gv.scan_files`), so the build uses h5py (parser b), as the plan said: keep the faster.

### The source fill value does not match the land values (repair R21)

In every file sampled (2001–2024, all three experiments), each variable's `_FillValue` attribute
is float32(1.267651e30), but land is written as 2**100, 3 ulps away. CF readers mask by exact
equality, so xarray opening a source file leaves land unmasked at 1.27e30. The store declares
2**100 instead (`gv.FILL`), as Eli decided on 2026-11-19.

In [5]:
from obspec_utils.readers import BlockStoreReader
with h5py.File(BlockStoreReader(store, "data/2022/023_archv.2022_100_10_2d.nc")) as f:
    attr, ssh0 = f["ssh"].attrs["_FillValue"][0], f["ssh"][0]
land = ssh0[ssh0 > 1e29]
print("attribute", repr(attr), "bits", attr.view("u4"), "| land", np.unique(land).tolist(), "bits", np.unique(land.view("u4")).tolist())
print("points equal to the attribute:", int((ssh0 == attr).sum()), "| equal to 2**100:", int((ssh0 == gv.FILL).sum()), "of", ssh0.size)

attribute np.float32(1.267651e+30) bits 1904214019 | land [1.2676506002282294e+30] bits [1904214016]
points equal to the attribute: 0 | equal to 2**100: 945478 of 3229237


In [6]:
fp = report.dropna(subset=["fingerprint"]).groupby("fingerprint").key.apply(list)
for f, ks in fp.items():
    print(f, len(ks), ks[0])
lon = pd.DataFrame({k: s["coords_md5"] for k, s in scans.items()}).T
print(lon.groupby(["Latitude", "Longitude"], dropna=False).size())

615cdf11fc 1 2005/010_archv.2005_079_04_3z.nc
936320f7b7 15 2001/010_archv.2001_016_00_3z.nc
9eacfbca62 15 2001/010_archv.2001_016_00_2d.nc
Latitude  Longitude
ee434938  8edb2d05      6
          e3b00e34     25
dtype: int64


### R22: the per-hour `cell_methods` flag, on the sample

Each hour's flag comes from its file's own `cell_methods` attribute. The 2005-03-20 04Z 3z file, which HYCOM
rebuilt with `ncra`, must be flagged 1 (`MT_mean`) and its 2d file 0. The skipped 2004-06-01 19Z 2d file must be 2.

In [7]:
sample_times = pd.DatetimeIndex(sorted(sample.time.unique()))
flags = gv.cell_methods_flags(sample_times, sample, scans)
f = pd.DataFrame(flags, index=sample_times)
print(f.to_string())
assert f.loc["2005-03-20 04:00", "3z"] == 1 and f.loc["2005-03-20 04:00", "2d"] == 0
assert f.loc["2004-06-01 19:00", "2d"] == 2 and f.loc["2004-06-01 19:00", "3z"] == 0
assert ((f.drop(index=[pd.Timestamp("2005-03-20 04:00"), pd.Timestamp("2004-06-01 19:00")]) == 0).all().all())

                     2d  3z
2001-01-16 00:00:00   0   0
2001-02-06 00:00:00   0   0
2002-04-12 05:00:00   0   0
2003-10-01 00:00:00   0   0
2003-10-29 06:00:00   0   0
2004-06-01 19:00:00   2   0
2005-03-20 04:00:00   0   1
2017-06-01 18:00:00   0   0
2017-06-01 19:00:00   0   0
2020-01-20 00:00:00   0   0
2022-04-10 12:00:00   0   0
2023-12-31 00:00:00   0   0
2024-01-01 18:00:00   0   0
2024-01-01 19:00:00   0   0
2024-02-15 17:00:00   0   0
2024-04-28 23:00:00   0   0


## 4. Build: six hours of 2022, two 3z chunkings

2022-04-10 10Z to 15Z, all experiment 023. Two gaps are made deliberately, one of each real kind:
- **13Z: 3z left out, 2d kept.** This is like the 41 layout A hours with a bad or missing 3z file.
- **14Z: both left out.** This is like an hour not yet copied.

Written as production will write it: a skeleton commit (coordinates, `bathy`, `mask`, empty virtual
arrays over the whole axis), then one region write and commit per batch, here two batches of three
hours. Manifests are split every 3 steps, so the split is exercised too.

In [8]:
times = gv.hourly_axis(pd.Timestamp("2022-04-10 10:00"), pd.Timestamp("2022-04-10 15:00"))
gap_3z, gap_both = pd.Timestamp("2022-04-10 13:00"), pd.Timestamp("2022-04-10 14:00")
smoke = files[files.time.isin(times)]
smoke = smoke[~((smoke.time == gap_3z) & (smoke.kind == "3z")) & (smoke.time != gap_both)]
print(smoke[["key", "kind"]].to_string(index=False))

t0 = time.perf_counter()
smoke_scans, smoke_report = gv.scan_files(smoke, "h5py", workers=8)
assert smoke_report.error.isna().all(), smoke_report
print(f"scanned {len(smoke)} files in {time.perf_counter() - t0:.1f} s")

ref2d, ref3z = "2022/023_archv.2022_100_10_2d.nc", "2022/023_archv.2022_100_10_3z.nc"
grid = gv.read_grid(store, ref2d, ref3z)
topo = gv.read_topo(store, grid)
src_attrs = {**smoke_scans[ref2d]["attrs"], **smoke_scans[ref3z]["attrs"]}
gattrs = gv.global_attrs(smoke_scans[ref2d]["global_attrs"])

                             key kind
2022/023_archv.2022_100_10_2d.nc   2d
2022/023_archv.2022_100_10_3z.nc   3z
2022/023_archv.2022_100_11_2d.nc   2d
2022/023_archv.2022_100_11_3z.nc   3z
2022/023_archv.2022_100_12_2d.nc   2d
2022/023_archv.2022_100_12_3z.nc   3z
2022/023_archv.2022_100_13_2d.nc   2d
2022/023_archv.2022_100_15_2d.nc   2d
2022/023_archv.2022_100_15_3z.nc   3z


scanned 9 files in 2.9 s


In [9]:
workdir = Path(tempfile.mkdtemp(prefix="gom-smoke-"))
built = {}
for d in (10, 1):
    path = workdir / f"d{d}"
    repo = icechunk.Repository.create(icechunk.local_filesystem_storage(str(path)), gv.repository_config(steps_per_manifest=3))
    repo.save_config()      # so a reader finds the virtual chunk container
    t0 = time.perf_counter()
    skeleton = xr.merge([gv.loaded_variables(times, smoke, grid, topo, smoke_scans),
                         gv.virtual_variables(times, smoke.iloc[:0], {}, d, src_attrs)])
    skeleton.attrs = gattrs
    s = repo.writable_session("main")
    skeleton.vz.to_icechunk(s.store, group=gv.GROUP, mode="w")
    s.commit("skeleton")
    for a, b in ((0, 3), (3, 6)):
        batch = gv.virtual_variables(times[a:b], smoke, smoke_scans, d, src_attrs)
        s = repo.writable_session("main")            # a committed session is read-only
        batch.vz.to_icechunk(s.store, group=gv.GROUP, region={"MT": slice(a, b)})
        s.commit(f"references {times[a]:%H}-{times[b-1]:%H}Z")
    built[d] = path
    size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    print(f"d={d:2d}: built in {time.perf_counter() - t0:.1f} s; repository {size/1e6:.1f} MB on disk")

  2026-11-19T04:10:31.815318Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-19T04:10:31.816901Z  WARN icechunk_storage::readback: conditional PUT is enabled but `unsafe_use_metadata` is disabled — lost-response recovery for conditional writes requires user metadata to stamp write-ids; without it, transient PUT failures may surface as spurious conflicts even when the write actually landed. See icechunk_storage::Settings::unsafe_use_metadata.
    at icechunk-storage/src/readback.rs:31



d=10: built in 0.3 s; repository 8.1 MB on disk


  2026-11-19T04:10:32.087575Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



d= 1: built in 0.3 s; repository 8.2 MB on disk


## 5. Fresh read-only session

A new `Repository.open`, not the writer's object, with the reader's explicit authorization of the HTTPS prefix.

In [10]:
def open_store(path, **kw):
    repo = icechunk.Repository.open(icechunk.local_filesystem_storage(str(path)),
                                    authorize_virtual_chunk_access=gv.reader_authorization())
    return xr.open_zarr(repo.readonly_session("main").store, group=gv.GROUP, consolidated=False, chunks=None, **kw)

ds = {d: open_store(p) for d, p in built.items()}
raw = {d: open_store(p, mask_and_scale=False, decode_times=False) for d, p in built.items()}
ds[1]

  2026-11-19T04:10:32.399913Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-19T04:10:32.431892Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-19T04:10:32.463400Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-19T04:10:32.485049Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/proc

<xarray.Dataset> Size: 16GB
Dimensions:                (MT: 6, Latitude: 1537, Longitude: 2101, Depth: 40)
Coordinates:
  * MT                     (MT) datetime64[ns] 48B 2022-04-10T10:00:00 ... 20...
    Date                   (MT) float64 48B ...
    cell_methods_3z        (MT) int8 6B ...
    experiment             (MT) int16 12B ...
    cell_methods_2d        (MT) int8 6B ...
  * Latitude               (Latitude) float32 6kB 18.09 18.1 ... 31.95 31.96
  * Longitude              (Longitude) float32 8kB -98.0 -97.99 ... -77.01 -77.0
  * Depth                  (Depth) float32 160B 0.0 2.0 4.0 ... 4e+03 5e+03
Data variables: (12/13)
    v_barotropic_velocity  (MT, Latitude, Longitude) float32 78MB ...
    u                      (MT, Depth, Latitude, Longitude) float32 3GB ...
    u_barotropic_velocity  (MT, Latitude, Longitude) float32 78MB ...
    wnd_nwd                (MT, Latitude, Longitude) float32 78MB ...
    w_velocity             (MT, Depth, Latitude, Longitude) float32 3GB ...
    mixed_layer_thickness  (MT, Latitude, Longitude) float32 78MB ...
    ...                     ...
    ssh                    (MT, Latitude, Longitude) float32 78MB ...
    wnd_ewd                (MT, Latitude, Longitude) float32 78MB ...
    mask                   (Latitude, Longitude) int8 3MB ...
    v                      (MT, Depth, Latitude, Longitude) float32 3GB ...
    water_temp             (MT, Depth, Latitude, Longitude) float32 3GB ...
    bathy                  (Latitude, Longitude) float32 13MB ...
Attributes:
    Conventions:         CF-1.11
    title:               HYCOM-TSIS GOMb0.01 Gulf of Mexico reanalysis, hourly
    source:              HYCOM archive file
    experiment:          01.0
    history:             python
    source_Conventions:  CF-1.0
    source_title:        HYCOM
    references:          https://www.hycom.org/data/gomb0pt01/gom-reanalysis
    source_data:         s3://hycom-tsis-gom-reanalysis (us-west-2), NetCDF-4...
    comment:             Virtual Icechunk store: Zarr metadata and byte-range...

In [11]:
for d in built:
    x, r = ds[d], raw[d]
    assert dict(x.sizes) == {"MT": 6, "Depth": 40, "Latitude": 1537, "Longitude": 2101}, dict(x.sizes)
    assert set(x.data_vars) == set(gv.VARS_2D + gv.VARS_3Z + ("bathy", "mask")), set(x.data_vars)
    assert (x.MT.values == times.values).all()
    assert x.experiment.values.tolist() == [23, 23, 23, 23, 0, 23], x.experiment.values
    assert x.cell_methods_2d.values.tolist() == [0, 0, 0, 0, 2, 0] and x.cell_methods_3z.values.tolist() == [0, 0, 0, 2, 2, 0]  # R22
    assert r.water_temp.encoding["chunks"] == (1, d, 385, 526) and r.ssh.encoding["chunks"] == (1, 1537, 2101)
    assert r.MT.attrs["units"] == "days since 1900-12-31 00:00:00" and r.MT.dtype == "float64"

# MT as stored must equal the 2d files' own MT, bit for bit (the source convention, decided).
from obspec_utils.readers import BlockStoreReader
with h5py.File(BlockStoreReader(store, "data/" + ref2d)) as f:
    mt_file = f["MT"][0]
assert raw[1].MT.values[0] == mt_file, (raw[1].MT.values[0], mt_file)
print("MT stored", repr(raw[1].MT.values[0]), "== 2d file", repr(mt_file))
print("Date", raw[1].Date.values)
print("experiment", ds[1].experiment.values, "| Latitude/Longitude/Depth", ds[1].Latitude.size, ds[1].Longitude.size, ds[1].Depth.values[[0, -1]])

MT stored np.float64(44295.416666666664) == 2d file np.float64(44295.416666666664)
Date [20220410.41666667 20220410.45833333 20220410.5        20220410.54166667
 20220410.58333333 20220410.625     ]
experiment [23 23 23 23  0 23] | Latitude/Longitude/Depth 1537 2101 [   0. 5000.]


### Attributes against the repair table

In [12]:
x = raw[1]
no_valid_range = [v for v in x.variables if "valid_range" in x[v].attrs]
assert not no_valid_range, no_valid_range                                              # R7
assert x.ssh.attrs["standard_name"] == "sea_surface_height_above_geoid"                # R9
assert x.ssh.attrs["long_name"] == " sea surf. height  [01.0H]"                        # R8: verbatim
assert (x.salinity.attrs["units"], x.salinity.attrs["source_units"], x.salinity.attrs["long_name"]) == ("1e-3", "psu", "Sea Water Salinity")  # R10
assert x.water_temp.attrs["units_metadata"] == "temperature: on_scale"                 # R11
assert x.MT.attrs["standard_name"] == "time" and x.MT.attrs["units_metadata"] == "leap_seconds: unknown"  # R2, R19
assert "units" not in x.Date.attrs and x.Date.attrs["source_units"] == "day as %Y%m%d.%f"  # R4
assert x.bathy.attrs["standard_name"] == "sea_floor_depth_below_geoid" and x.mask.dtype == "int8"  # R13, R14
assert x.attrs["Conventions"] == "CF-1.11" and "_NCProperties" not in x.attrs           # R15, R17
assert x.ssh.attrs["ancillary_variables"] == "cell_methods_2d" and x.u.attrs["ancillary_variables"] == "cell_methods_3z"  # R22
assert {"Date", "experiment"} <= set(ds[1].ssh.coords) and {"Date", "experiment"} <= set(ds[1].water_temp.coords)  # R20
zg = zarr.open_group(icechunk.Repository.open(icechunk.local_filesystem_storage(str(built[1]))).readonly_session("main").store, path=gv.GROUP, mode="r")
for c in ("MT", "Depth", "Latitude", "Longitude", "Date", "experiment", "cell_methods_2d", "cell_methods_3z"):
    assert "_FillValue" not in zg[c].attrs, (c, dict(zg[c].attrs))       # R5 (CF 2.5.1), in the store itself
print("store: no _FillValue attribute on any coordinate; zarr fill_value of ssh", zg["ssh"].fill_value, "== 2**100:", zg["ssh"].fill_value == 2.0**100)
for v in ("ssh", "salinity", "MT", "Date", "experiment", "cell_methods_3z", "bathy", "mask"):
    print(f"{v:11s}", x[v].attrs)
print(x.attrs)

  2026-11-19T04:10:32.600835Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



store: no _FillValue attribute on any coordinate; zarr fill_value of ssh 1.2676506e+30 == 2**100: True
ssh         {'long_name': ' sea surf. height  [01.0H]', 'standard_name': 'sea_surface_height_above_geoid', 'units': 'm', 'ancillary_variables': 'cell_methods_2d', '_FillValue': 1.2676506002282294e+30}
salinity    {'long_name': 'Sea Water Salinity', 'standard_name': 'sea_water_salinity', 'units': '1e-3', 'ancillary_variables': 'cell_methods_3z', 'source_units': 'psu', '_FillValue': 1.2676506002282294e+30}
MT          {'long_name': 'time', 'units': 'days since 1900-12-31 00:00:00', 'calendar': 'standard', 'axis': 'T', 'standard_name': 'time', 'units_metadata': 'leap_seconds: unknown'}
Date        {'long_name': 'date (YYYYMMDD.fraction of day)', 'source_units': 'day as %Y%m%d.%f', 'C_format': '%13.4f', 'FORTRAN_format': '(f13.4)'}
experiment  {'long_name': 'experiment number in the source file names (0: no file for this hour)', 'flag_values': [0, 10, 23, 26], 'flag_meanings': 'no_source_

## Values against h5py

The reference values are read straight from the source files with h5py. The comparisons are raw (`mask_and_scale=False`), so they must be exact.

In [13]:
from obspec_utils.readers import BlockStoreReader
def h5(key, var, sel):
    with h5py.File(BlockStoreReader(store, "data/" + key)) as f:
        return f[var][sel]
k = lambda h, kind: f"2022/023_archv.2022_100_{h:02d}_{kind}.nc"   # noqa: E731
row, col = 600, 1000              # inside a tile; 3z tile edges are at 385, 770, 1155 and 526, 1052, 1578
box = (slice(300, 500), slice(450, 650))   # straddles four tiles
for d in built:
    r, timing = raw[d], {}
    t0 = time.perf_counter(); a = r.ssh.isel(MT=0).values; timing["ssh map"] = time.perf_counter() - t0
    assert np.array_equal(a, h5(k(10, "2d"), "ssh", 0))
    t0 = time.perf_counter(); a = r.water_temp.isel(MT=5, Depth=25).values; timing["water_temp map, 1 depth"] = time.perf_counter() - t0
    assert np.array_equal(a, h5(k(15, "3z"), "water_temp", (0, 25)))
    t0 = time.perf_counter(); a = r.salinity.isel(MT=1, Latitude=row, Longitude=col).values; timing["salinity profile, 40 depths"] = time.perf_counter() - t0
    assert np.array_equal(a, h5(k(11, "3z"), "salinity", (0, slice(None), row, col)))
    t0 = time.perf_counter(); a = r.u.isel(MT=2, Depth=3, Latitude=box[0], Longitude=box[1]).values; timing["u 200x200 box over 4 tiles"] = time.perf_counter() - t0
    assert np.array_equal(a, h5(k(12, "3z"), "u", (0, 3, *box)))
    t0 = time.perf_counter(); a = r.water_temp.isel(Depth=0, Latitude=row, Longitude=col).values; timing["water_temp point, 6 hours"] = time.perf_counter() - t0
    want = [h5(k(h, "3z"), "water_temp", (0, 0, row, col)) for h in (10, 11, 12, 15)]
    assert np.array_equal(a[[0, 1, 2, 5]], want) and (a[[3, 4]] == gv.FILL).all()
    print(f"d={d:2d}  " + "  ".join(f"{n}: {s:.2f}s" for n, s in timing.items()))
print("every value identical to h5py")

d=10  ssh map: 0.21s  water_temp map, 1 depth: 0.40s  salinity profile, 40 depths: 0.33s  u 200x200 box over 4 tiles: 0.20s  water_temp point, 6 hours: 0.28s


d= 1  ssh map: 0.10s  water_temp map, 1 depth: 0.35s  salinity profile, 40 depths: 0.18s  u 200x200 box over 4 tiles: 0.06s  water_temp point, 6 hours: 0.12s
every value identical to h5py


In [14]:
# The two gaps, decoded: 13Z has 2d and no 3z; 14Z has neither. Everything else is data where the sea is.
x = ds[1]
sea = x.mask.values == 1
for t, label in ((gap_3z, "13Z (3z left out)"), (gap_both, "14Z (both left out)"), (times[0], "10Z")):
    s = x.ssh.sel(MT=t).values; w = x.water_temp.sel(MT=t).isel(Depth=0).values
    print(f"{label:22s} ssh defined at {np.isfinite(s).sum():>9,} points, water_temp(0 m) at {np.isfinite(w).sum():>9,}")
assert np.isnan(x.water_temp.sel(MT=gap_3z).isel(Depth=0).values).all() and np.isfinite(x.ssh.sel(MT=gap_3z).values).sum() == sea.sum()
assert np.isnan(x.ssh.sel(MT=gap_both).values).all()
assert (np.isfinite(x.ssh.isel(MT=0).values) == sea).all(), "mask == 1 exactly where ssh is defined"
print("mask == 1 exactly where ssh is defined;", f"bathy {float(x.bathy.min())}–{float(x.bathy.max()):.1f} m, 0 on land:", bool((x.bathy.values[~sea] == 0).all()))

13Z (3z left out)      ssh defined at 2,283,759 points, water_temp(0 m) at         0
14Z (both left out)    ssh defined at         0 points, water_temp(0 m) at         0


10Z                    ssh defined at 2,283,759 points, water_temp(0 m) at 2,283,759


mask == 1 exactly where ssh is defined; bathy 0.0–6726.5 m, 0 on land: True


## 6. CF check

The checker reads NetCDF, so a small corner of the store, including both gap hours, is exported, as in the GOFS smoke test.

In [15]:
from compliance_checker.runner import ComplianceChecker, CheckSuite
cf_nc = workdir / "cf-check.nc"
sub = ds[1].isel(Depth=slice(0, 2), Latitude=slice(800, 804), Longitude=slice(1000, 1004)).load()
for v in sub.variables:   # Zarr-only encoding keys mean nothing to NetCDF
    sub[v].encoding = {k: e for k, e in sub[v].encoding.items() if k in ("dtype", "units", "calendar", "_FillValue")}
    if v in sub.dims:
        sub[v].encoding["_FillValue"] = None   # as in the store; xarray would otherwise add one on export
# Zarr attributes are JSON and carry no dtype; CF wants flag_values typed like the variable.
for v in ("experiment", "mask", "cell_methods_2d", "cell_methods_3z"):
    sub[v].attrs["flag_values"] = np.asarray(sub[v].attrs["flag_values"], sub[v].dtype)
sub.to_netcdf(cf_nc)
CheckSuite.load_all_available_checkers()
passed, errors = ComplianceChecker.run_checker(str(cf_nc), ["cf:1.11"], verbose=1, criteria="normal")
print(f"cf:1.11 passed: {passed}   checker errors: {errors}")
assert passed



--------------------------------------------------------------------------------
                         IOOS Compliance Checker Report                         
                                 Version 6.1.0                                  
                     Report generated 2026-11-19T04:10:48Z                      
                                    cf:1.11                                     
http://cfconventions.org/Data/cf-conventions/cf-conventions-1.11/cf-conventions.html
--------------------------------------------------------------------------------
All tests passed!
cf:1.11 passed: True   checker errors: False


## 7. What is still unknown

To review before the 2022 performance test (step 3):

- **Scan rate for the full archive.** h5py is serialised by a global lock, so the scan runs in
  processes. Measured on this 4-core hub: about 17 files/s at 16 processes. That is 17 min for 2022 and
  about 5 h for all 296,585 layout A files. A larger VM in us-west-2 should scale with processes. Measure in step 3.
- **Build cost at d = 1 for a month and a year:** memory of the reference arrays, region-write and
  commit times, and the manifest sizes. Six hours shows nothing about that.
- **Read timings here are local-repository, in-region, at `async.concurrency = 128`.** The step 3
  table (point series over a year, box over a year) is the real test.
- **R21 (fill value): decided (Eli, 2026-11-19).**
- **How many layout A files are like 2004_153_19 (layout B chunking) or the ncra stand-ins?** The scan
  reports them. The full count comes from the 2022 test and then the full archive.
- **gridlook with a grouped store and `MT`/`Depth` dims**: step 4.

## Clean up

Only the temporary local repositories.

In [16]:
shutil.rmtree(workdir)
print("removed", workdir)

removed /tmp/gom-smoke-zjzyizbt
